# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python tools for tabular data analysis.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

> **Note:** This notebook uses the Croissant `@id` fields to reference every record set, field, and column.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
dataset_metadata = dataset.metadata
print(f"{dataset_metadata.name}: {dataset_metadata.description}\n\nIdentifier: {dataset_metadata.identifier}")

## 2. Data Overview
Explore available record sets, fields, and their `@id`s. This allows you to programmatically select which data to process next.

We'll enumerate record sets and fields, always referencing their `@id` values as required for future extraction.

In [ ]:
# List available record sets, their @id and fields with @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for recset in record_sets:
    print(f"- RecordSet name: {recset.name} | @id: {recset['@id']}")
    for field in recset.fields:
        print(f"    - Field: {field.name} | @id: {field['@id']} | dataType: {getattr(field, 'dataType', None)}")

## 3. Data Extraction
Let's load all data from the record set(s) into DataFrames for inspection and analysis. **You must reference record set and field by their `@id`.**

We'll demonstrate this step with the first available record set, but you can extend to more if present.

In [ ]:
# Extract data from all record sets (referenced by their @id)
import collections

dataframes = {}

# Map: {<record_set @id>: <mlcroissant.records DataFrame>}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet @id={record_set_id} with shape {dataframes[record_set_id].shape}")
    else:
        print(f"Warning: No records found for RecordSet @id={record_set_id}")

# Show columns for the first DataFrame loaded (if any)
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nFields (@id) for RecordSet '{first_rs_id}':\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Now we perform basic filtering, normalization, and grouping. All operations are performed with reference to field `@id`s from earlier steps.

We'll choose one numeric field (for example: a patient's age or a biomarker value) and one categorical/grouping field from the record set. Update the variables as appropriate for your dataset's fields.

In [ ]:
# Select RecordSet and numeric field by @id (adjust as needed based on step 2 output)
# Example (replace with values from your record set):
example_record_set_id = first_rs_id  # Use the first record set loaded above
df = dataframes[example_record_set_id]

# Identify a numeric and a group field by @id -- edit as needed!
# If unsure, print df.dtypes to locate numerics and categoricals
print("DataFrame columns and types:\n", df.dtypes)

# --- USER: Replace these with actual @id values from your dataset ---
numeric_field_id = next((col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])), None)
group_field_id = next((col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id), None)

print(f"\nUsing numeric_field_id: {numeric_field_id}\nUsing group_field_id: {group_field_id}")

if numeric_field_id is not None:
    # Example filter: select records where numeric_field > threshold (edit as appropriate)
    threshold = df[numeric_field_id].mean()  # e.g., above average
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_zscore"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"\nNormalized (z-score) {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field (if available) and show group means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped average {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field, and the grouped mean as a barplot, using matplotlib or seaborn. These can be adjusted to your relevant columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (raw and normalized)
if numeric_field_id is not None:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    df[numeric_field_id].plot.hist(ax=axs[0], bins=15, color='skyblue', edgecolor='black')
    axs[0].set_title(f'Distribution of {numeric_field_id}')
    axs[0].set_xlabel(numeric_field_id)

    if f"{numeric_field_id}_zscore" in filtered_df.columns:
        filtered_df[f"{numeric_field_id}_zscore"].plot.hist(ax=axs[1], bins=15, color='salmon', edgecolor='black')
        axs[1].set_title(f'Normalized (z-score) {numeric_field_id}')
        axs[1].set_xlabel(f'{numeric_field_id}_zscore')
    else:
        axs[1].axis('off')

    plt.tight_layout()
    plt.show()

# Barplot of group average
if group_field_id in df.columns and numeric_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    group_means.plot(kind='bar', color='orchid', edgecolor='black')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library. We loaded metadata, enumerated record sets and fields using Croissant `@id` values, extracted data for analysis, performed filtering and normalization on a selected numeric field, grouped and visualized summary statistics.

**Key observations:**
- The dataset offers structured, schema-annotated insight into clinicopathological features of second primary colorectal cancer in cancer survivors.
- All references to data structures were made using Croissant `@id` values for clarity and reproducibility.
- The `mlcroissant` Python API enables seamless loading, inspection, and tabular processing of FAIR data.

> **Next steps:** Continue tailored analysis on chosen fields, cross-tabulations, or modeling as fits your research question.